In [0]:
# =============================================================================
# segmentation_full_reconciliation  -  READ ONLY. START-TO-FINISH proof that EVERY
# eligible case (raw AppealCase CaseType=1) lands in EXACTLY ONE bucket, with:
#   * ZERO droppage  (no case in raw but in no bucket = the 'Not sure?' gap)   -> RED FLAG
#   * ZERO duplication (no case in >1 bucket)                                  -> RED FLAG
#   * every bucket count == expected (Bella VM v6)                             -> RED FLAG on any diff
# Buckets: active (stg_segmentation_states) + FTA + UTA + FPA + TD (stg_*_filtered).
# NO TOLERANCE: the run PASSES only if orphans=0 AND duplication=0 AND every count matches
# AND raw_universe == distinct_union_of_buckets. Single print + downloadable Excel.
#
# WHY this exists: the earlier verification only checked overlaps + counts (presence-based),
# so 8 cases that fell to the SQL 'ELSE Not sure?' branch (in NO bucket) were invisible.
# This closes that hole by reconciling against the full raw universe.
# =============================================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
import io, base64, datetime, pandas as pd

# ---- config ----
RAW_TBL   = "hive_metastore.ariadm_active_appeals.raw_appealcase"   # universe: CaseType=1
CASETYPE  = 1
BUCKETS = {
 "active":"hive_metastore.ariadm_active_appeals.stg_segmentation_states",
 "FTA"   :"hive_metastore.ariadm_arm_fta.stg_appeals_filtered",
 "UTA"   :"hive_metastore.ariadm_arm_uta.stg_appeals_filtered",
 "FPA"   :"hive_metastore.ariadm_arm_fpa.stg_filepreservedcases_filtered",
 "TD"    :"hive_metastore.ariadm_arm_td.stg_td_filtered",
}
EXPECTED = { "active":6353, "FTA":116637, "UTA":8180, "FPA":429, "TD":1807980 }   # Bella VM v6 2026-08-20
ACTIVE_STATE_EXPECTED = {
 "appealSubmitted":43,"awaitingRespondentEvidence(a)":9,"awaitingRespondentEvidence(b)":42,
 "caseUnderReview":177,"decided(a)":1983,"decided(b)":21,"decision":331,"ended":419,
 "ftpaSubmitted(a)":301,"ftpaDecided":1231,"ftpaSubmitted(b)":124,"listing":436,
 "paymentPending":19,"prepareForHearing":1079,"reasonsForAppealSubmitted":109,"remitted":29,
}
KNOWN_DROP_MAX = 2   # ARIADM-376 (business to resolve). RED FLAG if orphans exceed this; listed regardless.
STATUS_TBL = "hive_metastore.ariadm_active_appeals.raw_status"   # to classify orphans (latest CaseStatus/Outcome)
FILELOC_TBL= "hive_metastore.ariadm_active_appeals.raw_filelocation"
# archive-internal overlap (esp TD vs FTA/UTA/FPA) is KNOWN-EXPECTED per ARIADM-2263: a case can have BOTH
# an appeal record (FTA/UTA/FPA) AND a Tribunal-Decision record (TD). 2307 only requires ACTIVE to be
# overlap-free. So the 2307 verdict gates on ACTIVE-involved overlaps == 0; archive-only overlaps are reported.
ALLOW_ARCHIVE_INTERNAL_OVERLAP = True

# --------------------------------------------------------------------------------------------------
# KNOWN-EXCEPTIONS REGISTER (2026-08-26) — accepted, explained items. A run PASSES (2307) if the ONLY
# deviations are these; anything NEW (active overlap, unexpected archive-overlap pair, an orphan that is
# neither UT-active nor an empty shell) is a RED flag. Update this register as the business decides.
EXPECTED_ARCHIVE_OVERLAPS = ["FTA+TD","TD+UTA","FPA+TD"]   # ARIADM-2263, Bella confirmed 2026-08-26
UT_STATUS_SET = ['40','41','42','43','44','45','53','27','28','29','34','32','33']  # Upper-Tribunal statuses
# orphan categories:
#   ut_active     = latest CaseStatus in UT_STATUS_SET  -> UT 'active' cases, PENDING BUSINESS DECISION on destination
#   empty_shell   = zero status rows AND zero filelocation rows -> no lifecycle, LEFT BEHIND (e.g. LR/00001/2021)
#   NEW           = anything else -> RED FLAG (investigate)
KNOWN_EMPTY_SHELLS = ["LR/00001/2021"]   # confirmed empty (no status, no filelocation)
# Confirmed LEFT BEHIND (2026-08-26): have status but fail every state's criteria — verified the PySpark MATCHES
# Bella's SQL for these combos (50|91, 46|31, 17|38); they fail the secondary previous-status (st/sa) condition
# in BOTH, e.g. the two 50|91 cases have only 1 status row (no prior status). Not a bug — genuinely unclassifiable.
KNOWN_LEFT_BEHIND = ["HU/00492/2025","HU/00493/2025","HU/06218/2019","DC/00099/2019","IA/17104/2015"]

REPORT=[]; log=lambda *a: REPORT.append(" ".join(str(x) for x in a))
def logdf(df,n=60):
    try: REPORT.append(df._jdf.showString(n,0,False))
    except Exception as e: REPORT.append(f"  (render fail: {str(e)[:120]})")
def col_ci(cols,name): return next((c for c in cols if c.lower()==name.lower()),None)
_c=spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_c.first()["env"].strip().lower(); lz_key=_c.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}",f"ingest{lz_key}raw{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")

In [0]:
# ---- load universe + buckets, build membership ----
raw=spark.table(RAW_TBL); r_cc=col_ci(raw.columns,"CaseNo"); r_ct=col_ci(raw.columns,"CaseType")
r_dp=col_ci(raw.columns,"DeptId") or col_ci(raw.columns,"DepartmentId"); r_cp=col_ci(raw.columns,"CasePrefix")
U=(raw.filter(col(r_ct)==CASETYPE).select(trim(col(r_cc)).alias("CaseNo")).dropDuplicates(["CaseNo"]))
uni_total=U.count()

def bucket_cases(tbl):
    t=spark.table(tbl); cc=col_ci(t.columns,"CaseNo")
    return t.select(trim(col(cc)).alias("CaseNo")).dropDuplicates(["CaseNo"])
memb=None
for b,tbl in BUCKETS.items():
    d=bucket_cases(tbl).withColumn("bucket",lit(b))
    memb=d if memb is None else memb.unionByName(d)
memb=memb.cache()
union_total=memb.select("CaseNo").distinct().count()

# per-case bucket count/set
per_case=memb.groupBy("CaseNo").agg(countDistinct("bucket").alias("nb"), collect_set("bucket").alias("buckets")).cache()

log("="*84); log("SEGMENTATION FULL RECONCILIATION  (zero-droppage, start->finish)"); log("="*84)
log(f"START  raw universe (CaseType={CASETYPE}) : {uni_total}")
log(f"END    distinct union of all buckets      : {union_total}")

In [0]:
# ---- 1) per-bucket count vs expected ----
log("\n"+"-"*84); log("1) PER-BUCKET COUNT vs EXPECTED (Bella VM v6)"); log("-"*84)
bucket_rows=[]
for b in BUCKETS:
    n=memb.filter(col("bucket")==b).select("CaseNo").distinct().count()
    exp=EXPECTED.get(b); diff=(n-exp) if exp is not None else None
    st="MATCH" if diff==0 else "*** DIFF ***"
    log(f"  {b:8s} actual={n:>9}  expected={exp:>9}  diff={diff:>+6}  {st}")
    bucket_rows.append({"bucket":b,"actual":n,"expected":exp,"diff":diff,"status":("MATCH" if diff==0 else "DIFF")})

# ---- 2) DUPLICATION: any case in >1 bucket -- SPLIT active-involved (hard gate) vs archive-only (expected) ----
dup=per_case.filter(col("nb")>1).withColumn("has_active",array_contains(col("buckets"),lit("active"))).cache()
ndup=dup.count()
n_active_dup=dup.filter(col("has_active")).count()          # <<< the 2307 gate: MUST be 0
n_arch_dup=ndup-n_active_dup                                 # archive-internal (TD vs appeals) = known-expected
log("\n"+"-"*84); log(f"2) DUPLICATION (case in >1 bucket) : {ndup} total"); log("-"*84)
log(f"   2a) ACTIVE-involved overlaps : {n_active_dup}   <<< MUST BE 0  (this is the ARIADM-2307 requirement)")
log(f"   2b) archive-only overlaps    : {n_arch_dup}   (TD vs appeals = KNOWN-EXPECTED per ARIADM-2263)")
dup_rows=[]; active_dup_rows=[]
if ndup:
    combo=dup.withColumn("combo",array_join(array_sort(col("buckets")),"+")).groupBy("combo").count().orderBy(desc("count"))
    log("   -- FULL bucket-combo breakdown (all overlaps, not a sample) --"); logdf(combo,60)
    # export EVERY active-involved overlap (should be none); sample the archive-only ones
    for r in dup.filter(col("has_active")).withColumn("combo",array_join(array_sort(col("buckets")),"+")).select("CaseNo","combo").collect():
        active_dup_rows.append({"CaseNo":r["CaseNo"],"in_buckets":r["combo"]})
    for r in dup.withColumn("combo",array_join(array_sort(col("buckets")),"+")).select("CaseNo","combo").limit(500).collect():
        dup_rows.append({"CaseNo":r["CaseNo"],"in_buckets":r["combo"]})
    if n_active_dup==0: log("   >>> CONFIRMED: ZERO active-involved overlaps across the FULL set (2307 overlaps resolved).")
    else:               log(f"   >>> *** {n_active_dup} ACTIVE overlaps remain - see active_overlaps sheet - 2307 NOT clean ***")

# ---- 3) ORPHAN / DROPPAGE: universe case in NO bucket ----
orph=U.join(memb.select("CaseNo").distinct(),"CaseNo","left_anti").cache()
norph=orph.count()
log("\n"+"-"*84); log(f"3) ORPHAN / DROPPED (in raw universe, in NO bucket) : {norph}   <<< MUST BE 0"); log("-"*84)
orph_rows=[]; orph_class_rows=[]
n_ut_active=n_empty_shell=n_new_orphan=n_left_behind=0
if norph:
    # RESTRICT every big table (raw_appealcase/status/filelocation) to the ~orphan CaseNos FIRST, broadcasting
    # the tiny orphan key set — else aggregating/joining the full tables triggers a broadcast-join OOM.
    orph_cn=orph.select("CaseNo").dropDuplicates(["CaseNo"]).cache()
    # base fields (raw_appealcase restricted to orphans; all orphans are in raw so inner join is safe)
    od=(F.broadcast(orph_cn).join(raw.select(trim(col(r_cc)).alias("CaseNo"),
                             col(r_ct).alias("CaseType"),
                             (col(r_cp) if r_cp else lit(None)).alias("CasePrefix")),"CaseNo","inner"))
    # status ROW COUNT + reliable LATEST status (use F.max, NOT the undefined spark_max that silently failed before)
    st=spark.table(STATUS_TBL); s_cc=col_ci(st.columns,"CaseNo"); s_sid=col_ci(st.columns,"StatusId")
    s_cs=col_ci(st.columns,"CaseStatus"); s_oc=col_ci(st.columns,"Outcome")
    w=(st.select(trim(col(s_cc)).alias("CaseNo"),col(s_sid).alias("sid"),col(s_cs).alias("CaseStatus"),col(s_oc).alias("Outcome"))
         .join(F.broadcast(orph_cn),"CaseNo","inner")).cache()   # only orphan status rows
    srows=w.groupBy("CaseNo").agg(F.count(lit(1)).alias("status_rows"))
    latest=w.groupBy("CaseNo").agg(F.max("sid").alias("sid")).join(w,["CaseNo","sid"],"left")\
            .select("CaseNo",col("CaseStatus").alias("latest_CaseStatus"),col("Outcome").alias("latest_Outcome"))
    # filelocation row count + DeptId (also restricted to orphans first)
    fl=spark.table(FILELOC_TBL); f_cc=col_ci(fl.columns,"CaseNo"); f_dp=col_ci(fl.columns,"DeptId") or col_ci(fl.columns,"DepartmentId")
    flo=(fl.select(trim(col(f_cc)).alias("CaseNo"),col(f_dp).alias("DeptId")).join(F.broadcast(orph_cn),"CaseNo","inner")).cache()
    frows=flo.groupBy("CaseNo").agg(F.count(lit(1)).alias("filelocation_rows"))
    fdept=flo.dropDuplicates(["CaseNo"])
    od=(od.join(srows,"CaseNo","left").join(latest,"CaseNo","left").join(frows,"CaseNo","left").join(fdept,"CaseNo","left")
          .withColumn("status_rows",F.coalesce(col("status_rows"),lit(0)))
          .withColumn("filelocation_rows",F.coalesce(col("filelocation_rows"),lit(0))))
    # CLASSIFY each orphan
    od=od.withColumn("orphan_class",
        when(col("latest_CaseStatus").cast("string").isin(UT_STATUS_SET), lit("ut_active"))
        .when(col("CaseNo").isin(KNOWN_LEFT_BEHIND), lit("left_behind"))
        .when((col("status_rows")==0) & (col("filelocation_rows")==0), lit("empty_shell"))
        .otherwise(lit("NEW"))).cache()
    n_ut_active   = od.filter(col("orphan_class")=="ut_active").count()
    n_left_behind = od.filter(col("orphan_class")=="left_behind").count()
    n_empty_shell = od.filter(col("orphan_class")=="empty_shell").count()
    n_new_orphan  = od.filter(col("orphan_class")=="NEW").count()
    log("   -- ORPHAN CLASSIFICATION --")
    log(f"     ut_active   (latest status in UT set) : {n_ut_active:>5}   = UT 'active' cases, PENDING BUSINESS DECISION")
    log(f"     left_behind (fail all state criteria) : {n_left_behind:>5}   = LEFT BEHIND (confirmed; PySpark matches SQL, {KNOWN_LEFT_BEHIND})")
    log(f"     empty_shell (no status + no fileloc)  : {n_empty_shell:>5}   = no lifecycle, LEFT BEHIND (e.g. {KNOWN_EMPTY_SHELLS})")
    log(f"     NEW / unexplained                     : {n_new_orphan:>5}   <<< RED FLAG if > 0 (investigate)")
    log("   -- orphan latest-status distribution --"); logdf(od.groupBy("latest_CaseStatus","latest_Outcome").count().orderBy(desc("count")),40)
    if n_new_orphan: log("   -- NEW/unexplained orphans (investigate) --"); logdf(od.filter(col("orphan_class")=="NEW").limit(100),100)
    for r in od.limit(1000).collect():
        orph_rows.append({"CaseNo":r["CaseNo"],"CasePrefix":r["CasePrefix"],"orphan_class":r["orphan_class"],
                          "status_rows":r["status_rows"],"latest_CaseStatus":r["latest_CaseStatus"],"latest_Outcome":r["latest_Outcome"],
                          "filelocation_rows":r["filelocation_rows"],"DeptId":r["DeptId"]})
    orph_class_rows=[{"orphan_class":"ut_active","count":n_ut_active,"disposition":"pending business decision"},
                     {"orphan_class":"left_behind","count":n_left_behind,"disposition":"left behind (confirmed)"},
                     {"orphan_class":"empty_shell","count":n_empty_shell,"disposition":"left behind"},
                     {"orphan_class":"NEW","count":n_new_orphan,"disposition":"RED - investigate"}]

# ---- 4) EXTRA: in a bucket but NOT in raw CaseType=1 universe ----
extra=memb.select("CaseNo").distinct().join(U,"CaseNo","left_anti").cache()
nextra=extra.count()
log("\n"+"-"*84); log(f"4) EXTRA (in a bucket but not in raw CaseType={CASETYPE}) : {nextra}   <<< should be 0"); log("-"*84)
extra_rows=[]
if nextra:
    ed=extra.join(memb,"CaseNo","left").groupBy("bucket").count().orderBy(desc("count"))
    logdf(ed,20)
    for r in extra.join(memb,"CaseNo","left").select("CaseNo","bucket").limit(200).collect():
        extra_rows.append({"CaseNo":r["CaseNo"],"in_bucket":r["bucket"]})

# ---- 5) active state breakdown vs expected + NOT-SURE detector ----
log("\n"+"-"*84); log("5) ACTIVE per-state vs expected  (+ 'Not sure?'/unexpected-state detector)"); log("-"*84)
act=spark.table(BUCKETS["active"]); a_ts=col_ci(act.columns,"TargetState")
state_rows=[]; notsure_rows=[]; n_notsure=0
if a_ts:
    ac=act.groupBy(col(a_ts).alias("state")).agg(countDistinct(trim(col(col_ci(act.columns,"CaseNo")))).alias("actual"))
    amap={r["state"]:r["actual"] for r in ac.collect()}
    for s,exp in ACTIVE_STATE_EXPECTED.items():
        a=amap.get(s,0); d=a-exp
        log(f"  {s:32s} actual={a:>6} expected={exp:>6} diff={d:>+5} {'MATCH' if d==0 else '*** DIFF ***'}")
        state_rows.append({"state":s,"actual":a,"expected":exp,"diff":d,"status":("MATCH" if d==0 else "DIFF")})
    # NOT-SURE: any active TargetState value NOT in the 16 expected (catches 'Not sure?'/ELSE/renamed/null)
    extra_states={k:v for k,v in amap.items() if k not in ACTIVE_STATE_EXPECTED}
    n_notsure=sum(extra_states.values()) if extra_states else 0
    log(f"\n  NOT-SURE / UNEXPECTED active states (outside the 16): {extra_states if extra_states else '(none)'}  -> {n_notsure} cases")
    if n_notsure==0: log("  >>> CONFIRMED: no active case in a 'Not sure?'/ELSE bucket (every active case has one of the 16 valid states).")
    else:            log("  >>> *** cases landed in an unexpected/Not-sure state - see not_sure sheet ***")
    for k,v in extra_states.items(): notsure_rows.append({"state":str(k),"count":v})

In [0]:
# ---- VERDICT + Excel + single print ----
counts_ok = all(r["status"]=="MATCH" for r in bucket_rows)
states_ok = all(r["status"]=="MATCH" for r in state_rows) if state_rows else True
recon_ok  = (uni_total==union_total)
# archive-overlap combos that are NOT in the expected register = unexpected
arch_combos=[r["in_buckets"] for r in dup_rows if "active" not in r["in_buckets"]]
unexpected_arch=sorted(set(c for c in arch_combos if c not in EXPECTED_ARCHIVE_OVERLAPS))
# GREEN: the core reconciliation
GREEN = (counts_ok and states_ok and n_active_dup==0 and n_notsure==0 and nextra==0)
# RED: anything NEW/unexplained (a new mismatch, active overlap, unexpected archive pair, or an orphan that is
#      neither UT-active nor an empty shell). If RED is empty and GREEN holds -> PASS (with known exceptions).
RED = (not GREEN) or (n_new_orphan>0) or (len(unexpected_arch)>0)
PASS_2307 = (GREEN and not RED)
# STRICT (informational): literally zero of everything incl the known-expected items
PASS_STRICT = (ndup==0 and norph==0 and nextra==0 and counts_ok and states_ok and recon_ok)

log("\n"+"="*84)
log("SEGMENTATION RECONCILIATION — SUMMARY (report page)")
log("="*84)
log("A) RECONCILED (all match):")
log(f"     bucket counts    : {'ALL MATCH' if counts_ok else '*** DIFF ***'}  (active {EXPECTED['active']} / FTA {EXPECTED['FTA']} / UTA {EXPECTED['UTA']} / FPA {EXPECTED['FPA']} / TD {EXPECTED['TD']})")
log(f"     active 16 states : {'ALL MATCH' if states_ok else '*** DIFF ***'}")
log(f"     active overlaps  : {n_active_dup}  ({'none' if n_active_dup==0 else 'RED'})")
log(f"     not-sure / ELSE  : {n_notsure}  ({'none' if n_notsure==0 else 'RED'})")
log(f"     extra (not in raw): {nextra}  ({'none' if nextra==0 else 'RED'})")
log("B) KNOWN & EXPECTED (explained — not defects):")
log(f"     TD<->appeals overlap : {n_arch_dup:>6}  = expected, a case can have both an appeal & a TD decision (ARIADM-2263, Bella confirmed)")
log(f"     UT-active orphans    : {n_ut_active:>6}  = UT 'active' cases (latest status in {UT_STATUS_SET}) — PENDING BUSINESS DECISION on destination")
log(f"     left-behind orphans  : {n_left_behind:>6}  = fail all state criteria (PySpark matches SQL) — LEFT BEHIND ({KNOWN_LEFT_BEHIND})")
log(f"     empty-shell orphans  : {n_empty_shell:>6}  = no status + no filelocation — LEFT BEHIND (e.g. {KNOWN_EMPTY_SHELLS})")
log("C) NEW / UNEXPLAINED (must be zero — RED if not):")
log(f"     new orphans          : {n_new_orphan:>6}  {'' if n_new_orphan==0 else '<<< RED - investigate'}")
log(f"     unexpected overlaps  : {len(unexpected_arch):>6}  {('- '+', '.join(unexpected_arch)) if unexpected_arch else ''}")
log("-"*84)
log(f">>> 2307 VERDICT: {'PASS - all numbers reconcile; remaining items are expected / business / left-behind, none are migration defects' if PASS_2307 else 'FAIL - see section C (new/unexplained)'}")
log(f"    (strict all-zero verdict, informational: {'PASS' if PASS_STRICT else 'FAIL - held only by the known-expected items in B'})")
log(f"    raw universe {uni_total}  =  union {union_total}  +  orphans {norph}   [orphans = {n_ut_active} UT-active + {n_left_behind} left-behind + {n_empty_shell} empty-shell + {n_new_orphan} new]")
log("="*84)
PASS=PASS_2307

import pandas as pd
def _pd(rows): return pd.DataFrame(rows) if rows else pd.DataFrame({"_":["(none)"]})
# report-page summary: A reconciled / B known-expected / C new. One row per line item with a RAG status.
report_rows=[
 {"section":"A RECONCILED","item":"bucket counts (active/FTA/UTA/FPA/TD)","value":("ALL MATCH" if counts_ok else "DIFF"),"rag":("GREEN" if counts_ok else "RED")},
 {"section":"A RECONCILED","item":"active 16 states","value":("ALL MATCH" if states_ok else "DIFF"),"rag":("GREEN" if states_ok else "RED")},
 {"section":"A RECONCILED","item":"active overlaps","value":n_active_dup,"rag":("GREEN" if n_active_dup==0 else "RED")},
 {"section":"A RECONCILED","item":"not-sure / ELSE active","value":n_notsure,"rag":("GREEN" if n_notsure==0 else "RED")},
 {"section":"A RECONCILED","item":"extra (in bucket not in raw)","value":nextra,"rag":("GREEN" if nextra==0 else "RED")},
 {"section":"B KNOWN & EXPECTED","item":"TD<->appeals overlap (ARIADM-2263)","value":n_arch_dup,"rag":"AMBER"},
 {"section":"B KNOWN & EXPECTED","item":"UT-active orphans (pending business decision)","value":n_ut_active,"rag":"AMBER"},
 {"section":"B KNOWN & EXPECTED","item":"left-behind orphans (confirmed, fail all state criteria)","value":n_left_behind,"rag":"AMBER"},
 {"section":"B KNOWN & EXPECTED","item":"empty-shell orphans (left behind)","value":n_empty_shell,"rag":"AMBER"},
 {"section":"C NEW / UNEXPLAINED","item":"new orphans","value":n_new_orphan,"rag":("GREEN" if n_new_orphan==0 else "RED")},
 {"section":"C NEW / UNEXPLAINED","item":"unexpected overlap pairs","value":"; ".join(unexpected_arch) or 0,"rag":("GREEN" if not unexpected_arch else "RED")},
 {"section":"VERDICT","item":"2307","value":("PASS" if PASS_2307 else "FAIL"),"rag":("GREEN" if PASS_2307 else "RED")},
]
summary=pd.DataFrame([{
 "raw_universe_CaseType1":uni_total, "distinct_union_of_buckets":union_total, "orphans_total":norph,
 "orphans_ut_active":n_ut_active, "orphans_left_behind":n_left_behind, "orphans_empty_shell":n_empty_shell, "orphans_NEW":n_new_orphan,
 "duplicated_total":ndup, "ACTIVE_overlaps":n_active_dup, "archive_TD_overlaps_expected":n_arch_dup,
 "unexpected_overlaps":len(unexpected_arch), "NOT_SURE_active":n_notsure, "extra_not_in_universe":nextra,
 "bucket_counts_all_match":counts_ok, "active_states_all_match":states_ok,
 "VERDICT_2307":("PASS" if PASS_2307 else "FAIL"), "VERDICT_STRICT":("PASS" if PASS_STRICT else "FAIL")}])
sheets={"summary_report":_pd(report_rows),"summary":summary,"per_bucket":_pd(bucket_rows),
        "active_overlaps":_pd(active_dup_rows),"not_sure":_pd(notsure_rows),
        "duplicated_sample":_pd(dup_rows),"orphan_classes":_pd(orph_class_rows),
        "orphans_detail":_pd(orph_rows),"extra":_pd(extra_rows),"active_states":_pd(state_rows)}
try:
    try: import openpyxl
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    buf=io.BytesIO()
    with pd.ExcelWriter(buf,engine="openpyxl") as xw:
        for nm,pdf in sheets.items(): pdf.to_excel(xw,sheet_name=nm[:31],index=False)
    b64=base64.b64encode(buf.getvalue()).decode(); stamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    displayHTML(f'<a download="segmentation_full_reconciliation_{stamp}.xlsx" style="display:inline-block;background:#0b5cad;color:#fff;text-decoration:none;padding:8px 14px;border-radius:6px;font-family:sans-serif;font-size:13px" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">&#11015; Download full reconciliation ({len(sheets)} sheets)</a>')
except Exception as e: log(f"(download note: {str(e)[:120]})")
full="\n".join(REPORT)
try:
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/segmentation_full_reconciliation/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    p=f"{folder}/segmentation_full_reconciliation.txt"; open(p,"w").write(full); full+=f"\n\n>>> saved to: {p}"
except Exception as e: full+=f"\n(save note: {str(e)[:80]})"
print(full)